In [ ]:
# !pip install peft
# !pip install accelerate
# !pip install bitsandBytes
# !pip install transformers
# !pip install datasets
# !pip install "protobuf<4"
# !pip install GPUtil
# !pip install wandb

In [ ]:
import torch
import GPUtil
import os


GPUtil.showUtilization()

if torch.cuda.is_available():
    print("GPU is available")
else:
    print("GPU is not available, using CPU instead")

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

| ID | GPU | MEM |
------------------
|  0 |  6% | 13% |
GPU is available


In [5]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, LlamaTokenizer
# from huggingface_hub import notebook_login
from datasets import load_dataset
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
import wandb
from dotenv import load_dotenv
load_dotenv()


True

In [6]:
# 1. Authenticate W&B via Kaggle Secrets

try:
    wandb_api_key = os.getenv("WANDB_API_KEY")
    wandb.login(key=wandb_api_key)
except Exception as e:
    print("Warning: WANDB_API_KEY not found in Kaggle Secrets.")

wandb.init(project="Verifi-Finance-LLM", name="kaggle-llama-run-1")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hope/.netrc
wandb: Currently logged in as: hopesuccess51 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# # !hf auth login

# if "COLAB_GPU" in os.environ:
#   !huggingface-cli login
# else:
#   notebook_login()

In [7]:
dataset = load_dataset("virattt/financial-qa-10K", split="train")

Generating train split: 100%|██████████| 7000/7000 [00:00<00:00, 119846.06 examples/s]


In [8]:
base_model_id = "meta-llama/Llama-3.2-1B"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config)

Loading weights: 100%|██████████| 146/146 [00:03<00:00, 39.77it/s]


In [9]:
dataset[0]

{'question': 'What area did NVIDIA initially focus on before expanding to other computationally intensive fields?',
 'answer': 'NVIDIA initially focused on PC graphics.',
 'context': 'Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.',
 'ticker': 'NVDA',
 'filing': '2023_10K'}

In [10]:

tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_fast=False, trust_remote_code=True, add_eos_token=True)

if tokenizer.pad_token is None:
  tokenizer.add_special_tokens({'pad_token': tokenizer.eos_token})

In [11]:
def formatting_prompts_func(example):
    text = (
        f"### Financial Context:\n{example['context']}\n\n"
        f"### Question:\n{example['question']}\n\n"
        f"### Verified Answer:\n{example['answer']}"
    )
    return {"text": text}

formatted_dataset = dataset.map(formatting_prompts_func)

Map: 100%|██████████| 7000/7000 [00:00<00:00, 10256.99 examples/s]


In [12]:
formatted_dataset[0]

{'question': 'What area did NVIDIA initially focus on before expanding to other computationally intensive fields?',
 'answer': 'NVIDIA initially focused on PC graphics.',
 'context': 'Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.',
 'ticker': 'NVDA',
 'filing': '2023_10K',
 'text': '### Financial Context:\nSince our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.\n\n### Question:\nWhat area did NVIDIA initially focus on before expanding to other computationally intensive fields?\n\n### Verified Answer:\nNVIDIA initially focused on PC graphics.'}

In [13]:
tokenized_train_dataset = []
for phrase in formatted_dataset:
  tokenized_train_dataset.append(tokenizer(phrase["text"]))

tokenized_train_dataset[0]

{'input_ids': [14711, 17961, 9805, 512, 12834, 1057, 4113, 5357, 389, 6812, 14515, 11, 584, 617, 17626, 311, 3892, 1023, 3544, 323, 3062, 3801, 30154, 37295, 5151, 382, 14711, 16225, 512, 3923, 3158, 1550, 34661, 15453, 5357, 389, 1603, 24050, 311, 1023, 3801, 30154, 37295, 5151, 1980, 14711, 64269, 22559, 512, 45, 30452, 15453, 10968, 389, 6812, 14515, 13, 128001], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [14]:
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=8,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
    
)

model = get_peft_model(model, config)

In [ ]:
output_dir="../finetunedModel"

trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_train_dataset,
    args= transformers.TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=1e-4,
        # max_steps=20,
        bf16=False,
        optim="paged_adamw_8bit",
        logging_dir="./log",

        # W&B Logging
        report_to="wandb",
        logging_steps=10,
        
        # Checkpointing
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2, # Keep only 2 to avoid exceeding Kaggle's 20GB disk limit

),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
model.config.use_cache=False
trainer.train()




# # 3. Resume Logic
# resume_flag = True if os.path.exists(output_dir) and len(os.listdir(output_dir)) > 0 else False

# trainer.train(resume_from_checkpoint=resume_flag)

wandb.finish()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.369173
20,1.478738


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss


train/epoch,▁▇▇▁▇▇█
train/global_step,▁▇▇▁▇▇█
train/grad_norm,▆▁█▅
train/learning_rate,█▁█▁
train/loss,█▄▁▃
total_flos,62849563140096.0
train/epoch,0.012
train/global_step,21
train/grad_norm,2.622
train/learning_rate,1e-05
train/loss,1.47874


In [22]:
# 1. Define the final save path in Kaggle's permanent working directory
final_save_path = "/kaggle/working/Verifi-Finance-Adapter"

# 2. Save the trained LoRA adapter and model configuration
trainer.save_model(final_save_path)

# 3. Save the Tokenizer (Crucial!)
# Even though we didn't train the tokenizer, saving it ensures your 
# special templates (like the EOS token) are preserved for inference.
tokenizer.save_pretrained(final_save_path)

print(f"Model and tokenizer successfully saved to {final_save_path}")

Model and tokenizer successfully saved to /kaggle/working/Verifi-Finance-Adapter


Click the link below to download your model:


/kaggle/working/Verifi-Adapter.zip